# 16 — UR5e kinematics and Jacobians

**Objectives.** Compose six DH transforms, extract tool position/orientation, compute the geometric Jacobian, and visualize the arm. The model uses published geometry but conservative tutorial limits. It does **not** represent torque-level rigid-body dynamics or a safety-rated controller.


In [ ]:
import casadi as ca
import numpy as np
import matplotlib.pyplot as plt
from casadi_tutorial.models.ur5e import UR5eParameters,forward_kinematics,geometric_jacobian,joint_positions
from casadi_tutorial.plotting import plot_ur5e
p=UR5eParameters(); q=ca.SX.sym('q',6)
fk=ca.Function('fk',[q],[forward_kinematics(q,p)])
tool_jacobian=ca.Function('tool_jacobian',[q],[geometric_jacobian(q,p)])
q0=np.zeros(6); T=np.asarray(fk(q0).full()); J=np.asarray(tool_jacobian(q0).full())
print('base-to-tool at q=0:\n',T,'\nJacobian rank:',np.linalg.matrix_rank(J))
assert T.shape==(4,4) and J.shape==(6,6)
plot_ur5e(q0); plt.title('UR5e frame-origin stick model');


In [ ]:
direction=np.array([0.2,-0.1,0.3,0.1,-0.2,0.05]); eps=1e-6
fd=(np.asarray(fk(q0+eps*direction).full())[:3,3]-np.asarray(fk(q0-eps*direction).full())[:3,3])/(2*eps)
np.testing.assert_allclose(fd,J[:3]@direction,rtol=1e-6,atol=1e-8)


The top Jacobian block maps joint velocity to tool linear velocity; the bottom maps to angular velocity. Loss of rank indicates a kinematic singularity. Orientation errors should use a geometrically meaningful representation rather than subtracting rotation-matrix entries blindly.

**Common mistakes:** mixing standard/modified DH tables, millimetres/metres, degrees/radians, and frame conventions. **Recap:** validate a symbolic Jacobian with directional finite differences and known poses before optimizing.
